# 02 — Preprocessing
Clean, encode, scale, and split the dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

data_path = Path('../../data/plant_health_data.csv')
if not data_path.exists():
    data_path = Path('../data/plant_health_data.csv')

df = pd.read_csv(data_path)


In [2]:
# Drop timestamp but keep Plant_ID for group-aware evaluation
df = df.drop(columns=['Timestamp'])
print("Columns after drop:", df.columns.tolist())
print("Unique plants:", df['Plant_ID'].nunique())


Columns after drop: ['Plant_ID', 'Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature', 'Humidity', 'Light_Intensity', 'Soil_pH', 'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level', 'Chlorophyll_Content', 'Electrochemical_Signal', 'Plant_Health_Status']
Unique plants: 10


In [3]:
# Handle missing values
print("Missing before:", df.isnull().sum().sum())
df = df.dropna()
print("Missing after:", df.isnull().sum().sum())

Missing before: 0
Missing after: 0


In [4]:
# Encode target label with logical severity order
label_map = {'Healthy': 0, 'Moderate Stress': 1, 'High Stress': 2}
class_names = np.array(['Healthy', 'Moderate Stress', 'High Stress'])
df['Label'] = df['Plant_Health_Status'].map(label_map)

# Keep a LabelEncoder-compatible object for inverse_transform in later notebooks
le = LabelEncoder()
le.classes_ = class_names

print("Label mapping:", label_map)
df.head()


Label mapping: {'Healthy': 0, 'Moderate Stress': 1, 'High Stress': 2}


,Plant_ID,Soil_Moisture,Ambient_Temperature,Soil_Temperature,Humidity,Light_Intensity,Soil_pH,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Chlorophyll_Content,Electrochemical_Signal,Plant_Health_Status,Label
0,1,27.521109,22.240245,21.900435,55.291904,556.172805,5.581955,10.003650,45.806852,39.076199,35.703006,0.941402,High Stress,2
1,1,14.835566,21.706763,18.680892,63.949181,596.136721,7.135705,30.712562,25.394393,17.944826,27.993296,0.164899,High Stress,2
2,1,17.086362,21.180946,15.392939,67.837956,591.124627,5.656852,29.337002,27.573892,35.706530,43.646308,1.081728,High Stress,2
3,1,15.336156,22.593302,22.778394,58.190811,241.412476,5.584523,16.966621,26.180705,26.257746,37.838095,1.186088,High Stress,2
4,1,39.822216,28.929001,18.100937,63.772036,444.493830,5.919707,10.944961,37.898907,37.654483,48.265812,1.609805,High Stress,2


In [5]:
# Features, target, and grouping key
feature_columns = [col for col in df.columns if col not in ['Plant_Health_Status', 'Label', 'Plant_ID']]
X = df[feature_columns]
y = df['Label']
groups = df['Plant_ID']

print("Features:", X.shape)
print("Target:", y.shape)
print("Plants:", groups.nunique())


Features: (1200, 11)
Target: (1200,)
Plants: 10


In [6]:
# Pick the most class-representative plant holdout fold
overall_distribution = (
    y.value_counts(normalize=True)
    .sort_index()
    .reindex(sorted(label_map.values()), fill_value=0)
)
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
candidate_splits = []

for fold_id, (candidate_train_idx, candidate_test_idx) in enumerate(
    outer_splitter.split(X, y, groups), start=1
):
    fold_distribution = (
        y.iloc[candidate_test_idx]
        .value_counts(normalize=True)
        .sort_index()
        .reindex(overall_distribution.index, fill_value=0)
    )
    distribution_gap = np.abs(fold_distribution - overall_distribution).sum()
    candidate_splits.append((distribution_gap, fold_id, candidate_train_idx, candidate_test_idx))

selected_gap, selected_fold, train_idx, test_idx = min(candidate_splits, key=lambda item: item[0])

X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
train_groups = groups.iloc[train_idx].to_numpy()
test_groups = groups.iloc[test_idx].to_numpy()

print("Selected holdout fold:", selected_fold)
print("Holdout distribution gap:", round(float(selected_gap), 4))
print("Train plants:", sorted(pd.unique(train_groups).tolist()))
print("Test plants:", sorted(pd.unique(test_groups).tolist()))
print("Train class counts:", y_train.value_counts().sort_index().to_dict())
print("Test class counts:", y_test.value_counts().sort_index().to_dict())


Selected holdout fold: 2
Holdout distribution gap: 0.015
Train plants: [1, 2, 3, 5, 6, 8, 9, 10]
Test plants: [4, 7]
Train class counts: {0: 241, 1: 320, 2: 399}
Test class counts: {0: 58, 1: 81, 2: 101}


In [7]:
# Scale features using training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)
X_scaled = scaler.transform(X)

print("Scaling done. Train mean ~0:", X_train.mean(axis=0).round(2))
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Scaling done. Train mean ~0:

 [ 0. -0. -0.  0.  0. -0.  0. -0. -0.  0. -0.]
Train size: (960, 11)
Test size: (240, 11)


In [8]:
import joblib, os
os.makedirs('../../models', exist_ok=True)

# Save processed data, scaler, and split metadata
processed = pd.DataFrame(X_scaled, columns=X.columns)
processed['Label'] = y.values
processed.to_csv('../../data/processed_dataset.csv', index=False)

split_metadata = {
    'feature_names': feature_columns,
    'selected_holdout_fold': selected_fold,
    'holdout_distribution_gap': float(selected_gap),
    'train_indices': train_idx,
    'test_indices': test_idx,
    'train_groups': train_groups,
    'test_groups': test_groups,
    'train_plant_ids': sorted(pd.unique(train_groups).tolist()),
    'test_plant_ids': sorted(pd.unique(test_groups).tolist()),
}

joblib.dump(scaler, '../../models/scaler.pkl')
joblib.dump((X_train, X_test, y_train, y_test), '../../models/splits.pkl')
joblib.dump(split_metadata, '../../models/split_metadata.pkl')
joblib.dump(le, '../../models/label_encoder.pkl')
print("Saved: processed_dataset.csv, scaler.pkl, splits.pkl, split_metadata.pkl, label_encoder.pkl")


Saved: processed_dataset.csv, scaler.pkl, splits.pkl, split_metadata.pkl, label_encoder.pkl
